# LatentMAS-interp — GPU Test Suite

Pulls the repo, installs deps, runs logic tests + end-to-end smoke on real GPU.

In [ ]:
# 1. Clone / update repo
import os
REPO = "https://github.com/spraldev/LatentMAS-interp"
REPO_DIR = "/workspace/LatentMAS-interp"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO} {REPO_DIR}

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
# 2. Install deps
!pip install -q torch transformers accelerate datasets tqdm numpy

In [ ]:
# 3. Confirm GPU is visible
import torch
assert torch.cuda.is_available(), "No GPU found — wrong runtime?"
print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 4. Logic unit tests (no model, pure math/IO — should be <30s)
!python test_pipeline.py -v

In [ ]:
# 5. End-to-end smoke on GPU — downloads Qwen3-0.6B once (~2 GB), runs 5 ex/task (~5 min)
# This is the real test: actual model forward pass, W_a, KV capture, bucket assignment
import os, tempfile
SMOKE_DIR = "/workspace/runs/smoke"
os.makedirs(SMOKE_DIR, exist_ok=True)

!python final_run.py \
    --output_dir {SMOKE_DIR} \
    --conditions latent_mas single_agent_latent_greedy \
    --tasks gsm8k \
    --smoke --test

In [ ]:
# 6. Check smoke outputs
import json
from pathlib import Path

smoke = Path(SMOKE_DIR)

# W_a sanity
import torch
wa = torch.load(smoke / "wa_matrix.pt", weights_only=False)["W_a"].float()
S = torch.linalg.svdvals(wa)
frob = (wa - torch.eye(wa.shape[0])).norm().item()
print(f"W_a shape: {tuple(wa.shape)}")
print(f"cond number: {S.max()/S.min():.4f}  (want >>1 on Qwen3-4B, =1 on 0.6B/1.7B)")
print(f"Frob dist from identity: {frob:.4f}  (want >>1 on Qwen3-4B)")
print()

# example outputs
for cond in ["latent_mas", "single_agent_latent_greedy"]:
    metas = list((smoke / cond / "gsm8k").glob("example_*/metadata.json"))
    correct = sum(json.loads(m.read_text()).get("correct", False) for m in metas)
    print(f"{cond}: {correct}/{len(metas)} correct")

# buckets
bucket_file = smoke / "buckets" / "gsm8k.json"
if bucket_file.exists():
    from collections import Counter
    rows = json.loads(bucket_file.read_text())
    c = Counter(r["bucket"] for r in rows)
    print(f"\nBuckets: B1={c[1]} B2={c[2]} B3={c[3]} B4={c[4]} (n={len(rows)})")
    print(f"Bucket-1 rate: {c[1]/len(rows)*100:.1f}%  (need >0 on real 4B run)")
else:
    print("No bucket file yet — run both conditions first")